# Step 1 — Problem Observation & Setup Imports
### Credit Card Underwriting — End-to-End ML Pipeline
**Dataset:** `cc_underwriting_5k_stratified11.csv` | **Target:** `target_approved` (Yes / No)

---

## What is Underwriting?

Underwriting is the process a bank uses to decide **whether to approve or decline** a credit card application.

> **Real-World Analogy:** Imagine a bank manager who reviews every application, checks your income,
> credit score, debts, and spending history, then either hands you a card or says "sorry, not today."
> Our model *automates* that decision using historical data.

---

## The Business Problem

- **Goal:** Predict `target_approved` (Yes/No) for each applicant
- **Why it matters:** Approving risky applicants → credit losses; declining safe ones → lost revenue
- **Data:** 4,480 applicants × 200 features covering demographics, credit history, income, assets, fraud signals
- **Challenge:** Class imbalance (~65% Approved, ~35% Declined) + 200 raw features (need selection)

---

## End-to-End Pipeline Overview

| Notebook | Step | Description |
|----------|------|-------------|
| **01** | Problem & Setup | You are here — understand the problem, install libraries |
| **02** | EDA | Explore data shape, distributions, class balance |
| **03** | Missing Values | Detect and treat missing data |
| **04** | Data Cleaning | Outliers, encoding, type fixes, cleaning strategies |
| **05** | Feature Engineering | WoE, IV, interaction features, log transforms, multicollinearity |
| **06** | Train/Test Split | Stratified split, SMOTE, scaling |
| **07** | Model Selection | Why Random Forest? Baseline + tuning |
| **08** | Confusion Matrix | Deep-dive into the confusion matrix |
| **09** | Model Evaluation | AUC, Gini, KS, SHAP, scorecard banding |

---

## Key Terms You Will See Throughout

| Term | Plain-English Meaning |
|------|-----------------------|
| **Target** | The column we want to predict (Approved / Declined) |
| **Feature** | Any column used as input to the model |
| **AUC** | How well the model separates approvals from declines (0.5 = random, 1.0 = perfect) |
| **SMOTE** | Technique to balance the dataset by creating synthetic minority-class examples |
| **WoE** | Weight of Evidence — transforms features onto a log-odds scale |
| **IV** | Information Value — measures predictive power of a feature |
| **SHAP** | Explains *why* the model made each individual prediction |

---

## How to Run These Notebooks

1. Run notebooks **in order** (01 → 09)
2. Variables created in one notebook are **re-created at the top** of each subsequent one — no hidden state dependency
3. Every code cell has **comments on every meaningful line** explaining *what* and *why*


## 1.1 — Install Required Libraries

Before importing anything, we need to install the libraries not included in the standard Python distribution.

### What we are installing and why:

| Library | Purpose | Why We Need It |
|---------|---------|----------------|
| `shap` | Model explainability | Explains individual predictions using game theory |
| `imbalanced-learn` | Class imbalance handling | Provides SMOTE to oversample the minority class |

> All other libraries (`pandas`, `numpy`, `sklearn`, `matplotlib`, `seaborn`) come pre-installed in standard data science environments like Anaconda or Google Colab.


In [ ]:
# Install extra libraries not included by default
# The -q flag runs pip in quiet mode (less output clutter)
# These only need to be installed once per environment
!pip install -q shap imbalanced-learn

## 1.2 — Import All Libraries

We import everything at the top so:
- It is clear which libraries this project depends on
- If an import fails, we catch it early before running hours of code
- Each import is grouped by its role (core, visualisation, ML, etc.)


In [ ]:
# ── Core Python ───────────────────────────────────────────────────────────────
import warnings          # suppress non-critical deprecation warnings
import os                # file path operations

warnings.filterwarnings('ignore')   # hide warnings so output stays clean

# ── Data Manipulation ─────────────────────────────────────────────────────────
import numpy as np       # numerical arrays, math operations
import pandas as pd      # DataFrames — the main data structure we work with

# Display settings for nicer output
pd.set_option('display.max_columns', 50)           # show up to 50 columns
pd.set_option('display.float_format', '{:.4f}'.format)  # 4 decimal places

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt    # base plotting library
import matplotlib.ticker as mticker
import seaborn as sns              # higher-level, prettier plots built on matplotlib

sns.set_theme(style='whitegrid', palette='muted')  # clean background style
FIGSIZE = (14, 5)    # default figure size used throughout the project

# ── Scikit-Learn: Model Building ──────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,       # split data into train/test sets
    StratifiedKFold,        # K-fold keeping class ratio equal in each fold
    cross_val_score,        # evaluate model across multiple folds
    RandomizedSearchCV      # hyperparameter tuning via random search
)
from sklearn.preprocessing import (
    LabelEncoder,           # converts categorical strings to integers
    StandardScaler,         # scales features to zero-mean, unit-variance
    MinMaxScaler            # scales features to [0, 1] range
)
from sklearn.ensemble import RandomForestClassifier   # our main model
from sklearn.metrics import (
    roc_auc_score,          # AUC metric
    roc_curve,              # ROC curve data points
    classification_report,  # precision, recall, F1 for each class
    confusion_matrix,       # the 2x2 TP/TN/FP/FN matrix
    ConfusionMatrixDisplay, # visualise the confusion matrix
    precision_recall_curve  # precision vs recall tradeoff curve
)
from sklearn.impute import SimpleImputer   # fill missing values
from sklearn.pipeline import Pipeline     # chain preprocessing + model

# ── Imbalanced-Learn ──────────────────────────────────────────────────────────
# SMOTE = Synthetic Minority Oversampling Technique
# Creates synthetic examples of the minority class (Declined) to balance training data
from imblearn.over_sampling import SMOTE

# ── SHAP — Model Explainability ───────────────────────────────────────────────
# SHAP (SHapley Additive exPlanations) quantifies each feature's contribution
# to every individual prediction — essential for regulatory compliance in finance
import shap

# ── Reproducibility ───────────────────────────────────────────────────────────
# Setting a fixed random seed ensures every run produces identical results
# Without this, shuffles and random splits would differ each run
SEED = 42
np.random.seed(SEED)

print('All libraries loaded successfully')
print(f'  NumPy   version: {np.__version__}')
print(f'  Pandas  version: {pd.__version__}')
print(f'  Sklearn version: {__import__("sklearn").__version__}')

## 1.3 — Project Constants

Centralising constants makes the project easier to maintain.
Change a value once here — it updates everywhere automatically.


In [ ]:
# ── File Paths ────────────────────────────────────────────────────────────────
# Update DATA_PATH if your CSV is in a different location
DATA_PATH = 'cc_underwriting_5k_stratified11.csv'

# ── Columns to Exclude from Feature Lists ─────────────────────────────────────
# These columns are identifiers or alternative targets — never use as model inputs
IGNORE_COLS = ['applicant_id', 'target_approved', 'target_credit_limit_assigned']

# ── Key Numeric Columns for Initial EDA ───────────────────────────────────────
KEY_NUM = [
    'age', 'annual_income', 'fico_score', 'debt_to_income_ratio',
    'credit_utilization_ratio', 'net_worth', 'requested_credit_limit',
    'fraud_risk_score', 'avg_bureau_score', 'monthly_disposable_income'
]

# ── Thresholds ────────────────────────────────────────────────────────────────
HIGH_MISS_THRESH = 50.0   # drop columns with > 50% missing values
MIN_IV           = 0.02   # minimum IV to keep a feature (0.02 = "Weak" threshold)
CORR_THRESH      = 0.95   # drop one feature from any pair correlated above this

print('Project constants defined:')
print(f'  Data file     : {DATA_PATH}')
print(f'  Missing thresh: >{HIGH_MISS_THRESH}% -> drop column')
print(f'  Min IV        : {MIN_IV}')
print(f'  Corr thresh   : {CORR_THRESH}')

## Step 1 Complete

**What we did:**
- Understood the business problem (automate credit card approval decisions)
- Installed and imported all required libraries (annotated with purpose)
- Defined global constants and file paths

**Next:** `02_EDA.ipynb` — Explore the raw data: shape, distributions, class balance, column types
